# Clean-clone judge smoke (2026-09-15)

Reproduces what the judge does: `git clone` the fork's **main**, `pixi install`, then `eval.py` on all three RGB checkpoints.
Nothing from the training runtime is reused: the clone goes to a new folder and pixi builds its own Python 3.12 env (torch 2.12 / numpy 2.4 as pinned by upstream `pixi.toml`, not the 2.11 / 1.26 used for training).

Run cells one at a time. Logs and results sync to `MyDrive/marso/validations/clean_clone_smoke_<stamp>/`.


In [ ]:
import os, subprocess, hashlib, json, time
from pathlib import Path
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', timeout_ms=600000)
assert os.path.ismount('/content/drive')
STAMP = time.strftime('%Y%m%d_%H%M%S')
CLEAN = Path('/content/clean-clone') / STAMP
CLEAN.parent.mkdir(parents=True, exist_ok=True)
DRIVE_OUT = Path('/content/drive/MyDrive/marso/validations') / f'clean_clone_smoke_{STAMP}'
DRIVE_OUT.mkdir(parents=True, exist_ok=False)
LOG = DRIVE_OUT / 'log.txt'
ENV = dict(os.environ, DISPLAY='', PYOPENGL_PLATFORM='egl', PYTHONUNBUFFERED='1', PATH=os.path.expanduser('~/.pixi/bin') + ':' + os.environ['PATH'])
RECORD = {'stamp': STAMP, 'clean_dir': str(CLEAN), 'steps': []}

def run(cmd, label, cwd=None, quiet=False, timeout=3600, fatal=True):
    shell = isinstance(cmd, str)
    with LOG.open('a') as log:
        log.write(f'\n===== {label}: {cmd if shell else " ".join(map(str, cmd))}\n'); log.flush()
        t0 = time.time(); tail = []
        p = subprocess.Popen(cmd, cwd=cwd, env=ENV, shell=shell, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        n = 0
        for line in p.stdout:
            log.write(line); n += 1; tail.append(line); tail = tail[-30:]
            if not quiet or n % 25 == 0:
                print(line if not quiet else f'  … {n} lines, {time.time()-t0:.0f}s: {line.strip()[:110]}', end='' if not quiet else '\n', flush=True)
        rc = p.wait(timeout=timeout)
        log.write(f'===== {label}: exit {rc} in {time.time()-t0:.0f}s\n')
    RECORD['steps'].append({'label': label, 'exit': rc, 'seconds': round(time.time()-t0, 1)})
    (DRIVE_OUT / 'record.json').write_text(json.dumps(RECORD, indent=2))
    print(f'>>> {label}: exit {rc} ({time.time()-t0:.0f}s)', flush=True)
    if rc != 0:
        print(''.join(tail))
        if fatal: raise SystemExit(f'{label} failed (exit {rc})')
    return rc

def sha(p):
    with open(p, 'rb') as f: return hashlib.file_digest(f, 'sha256').hexdigest()

run(['git', 'clone', '--branch', 'main', 'https://github.com/mrpc2003/berlin-marso-hackathon.git', str(CLEAN)], 'git_clone')
head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=CLEAN, text=True, capture_output=True, check=True).stdout.strip()
print('HEAD', head); RECORD['head'] = head
expected = {'easy': '7a6de3d47039fb2cb8800b1b066c1c93bc045b1107692291925127786b2642b4',
            'medium': '35d47488b7f4dbf4b3f5eba55ba627fd6ca1b07d30c05cd88c6d49064b532210',
            'hard': '9dfd78d8b742a216d09c7c2acf3f60f2ab742c58809b3e7994102e932e03418c'}
import yaml
sub = yaml.safe_load((CLEAN / 'submission.yaml').read_text())
assert sub['rgb']['policy'] == 'warehouse_sort.il_policy:load_dp_rgb', sub
for level, digest in expected.items():
    rel = sub['rgb']['levels'][level]['checkpoint']
    actual = sha(CLEAN / rel)
    assert actual == digest, (level, rel, actual)
    print('CHECKPOINT_OK', level, rel, actual[:16])
RECORD['checkpoints_verified'] = expected
run('curl -fsSL https://pixi.sh/install.sh | bash', 'pixi_install_cli', quiet=True)
run(['pixi', '--version'], 'pixi_version', cwd=CLEAN)
run(['pixi', 'install'], 'pixi_install', cwd=CLEAN, quiet=True, timeout=5400)
rc = run(['pixi', 'run', 'install'], 'pixi_run_install', cwd=CLEAN, fatal=False)
RECORD['pixi_run_install_note'] = 'upstream pixi.toml has no install task; warehouse_sort is an editable pypi dependency installed by pixi install' if rc != 0 else 'ok'
run(['pixi', 'run', 'python', '-c', 'import torch, numpy, mani_skill, sapien, warehouse_sort, diffusers; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name() if torch.cuda.is_available() else None, "numpy", numpy.__version__, "mani_skill", mani_skill.__version__)'], 'pixi_env_probe', cwd=CLEAN)
print('CLEAN_CLONE_INSTALL_OK', str(CLEAN), str(DRIVE_OUT), flush=True)


In [ ]:
# Judge-style evaluation from the clean clone: 4-episode default config, then 32 episodes (seeds 5000+) per level.
RESULTS = DRIVE_OUT / 'results.jsonl'
for eval_cfg in ('conf/eval/default.yaml', 'conf/eval/eval32.yaml'):
    for level in ('easy', 'medium', 'hard'):
        ckpt = sub['rgb']['levels'][level]['checkpoint']
        run(['pixi', 'run', 'python', 'eval.py', f'difficulty={level}', 'obs_mode=rgb',
             f"policy={sub['rgb']['policy']}", f'checkpoint={ckpt}', f'eval_config={eval_cfg}',
             'record_video=false', f'results_file={RESULTS}'], f'eval_{Path(eval_cfg).stem}_{level}', cwd=CLEAN, quiet=True, timeout=3600)
print('CLEAN_CLONE_EVAL_DONE', str(RESULTS), flush=True)


In [ ]:
import json
rows = [json.loads(l) for l in (DRIVE_OUT / 'results.jsonl').read_text().splitlines() if l.strip()]
reference32 = {'easy': 1.0, 'medium': 0.984375, 'hard': 0.9635416666666666}  # training-runtime sweeps, same seeds, same winner settings
summary = []
for r in rows:
    cfg = Path(r['eval_config']).stem; lvl = r['level']
    ref = reference32.get(lvl) if cfg == 'eval32' else None
    row = {'eval_config': cfg, 'level': lvl, 'n_episodes': r['n_episodes'], 'sort_accuracy': r['sort_accuracy'],
           'all_placed_rate': r.get('all_placed_rate'), 'mis_sort_rate': r.get('mis_sort_rate'), 'eval_seconds': r.get('eval_seconds'),
           'reference_sort_accuracy': ref, 'abs_diff_vs_reference': (None if ref is None else round(abs(r['sort_accuracy'] - ref), 4))}
    summary.append(row)
    print(f"{cfg:8s} {lvl:6s} n={r['n_episodes']:3d} acc={r['sort_accuracy']:.4f} all_placed={r.get('all_placed_rate')} ref={ref} diff={row['abs_diff_vs_reference']}")
RECORD['summary'] = summary
(DRIVE_OUT / 'record.json').write_text(json.dumps(RECORD, indent=2))
print('CLEAN_CLONE_SMOKE_SUMMARY_WRITTEN', str(DRIVE_OUT / 'record.json'))
